In [6]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool 
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push

In [7]:
load_dotenv(override=True)

# Constants
MODEL_NAME = 'gpt-4o-mini'
USE_EMAIL = False
HOW_MANY_SEARCHES = 5

### Strategy for the Deep Research Agent  
We are going to do it the bulletproof way.  
We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.  
We will use Structured Outputs at each point.

### We will build 4 Agents:  
1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made  
3. The Writer Agent: writes a robust report  
4. The Emailer Agent: crafts and sends an email  

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.

### Agent 1: The Search Agent  
#### OpenAI Hosted Tools

https://openai.github.io/openai-agents-python/tools/#hosted-tools  

A paid, quick approach to carrying out managed functionality on OpenAI's cloud.  
Their docs surface these tools, but it's worth keeping in mind that they're costly and lock you in to the OpenAI ecosystem.


In [8]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""

task = "Most popular AI Agent framework in 2026"

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

In [9]:
search_agent = Agent(name='Search Agent', instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=settings)
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

As of September 2026, several AI agent frameworks have emerged as leaders in the field, each catering to specific needs and preferences. LangGraph stands out for its robust support of complex, stateful workflows, making it ideal for applications requiring durable execution and human-in-the-loop processes. Its graph-based orchestration allows for explicit state management and branching, which is particularly beneficial for intricate agentic tasks. ([the-agent-report.com](https://the-agent-report.com/2026/07/ai-agent-frameworks-comparison-2026-langgraph-crewai-autogen/?utm_source=openai))

For enterprises operating within Microsoft's ecosystem, the Microsoft Agent Framework 1.0 offers a comprehensive solution. This framework integrates concepts from AutoGen and Semantic Kernel, providing a unified platform for developing AI agents. It supports both Python and .NET, ensuring compatibility across various enterprise applications. ([the-agent-report.com](https://the-agent-report.com/2026/07/ai-agent-frameworks-comparison-2026-langgraph-crewai-autogen/?utm_source=openai))

OpenAI's Agents SDK is tailored for developers seeking a streamlined approach to building AI agents. It offers a clean and efficient environment for creating agents that can handle tasks such as tool calling, orchestration, and multi-agent coordination. Its simplicity and ease of use make it a popular choice for rapid prototyping and deployment. ([the-agent-report.com](https://the-agent-report.com/2026/07/ai-agent-frameworks-comparison-2026-langgraph-crewai-autogen/?utm_source=openai))

CrewAI focuses on facilitating the development of multi-agent systems through its role-based "crew" pattern. This framework enables the creation of collaborative agent teams with minimal code, making it suitable for applications that require coordinated efforts among multiple agents. ([the-agent-report.com](https://the-agent-report.com/2026/07/ai-agent-frameworks-comparison-2026-langgraph-crewai-autogen/?utm_source=openai))

Each of these frameworks offers unique features and advantages, allowing developers to choose the one that best aligns with their project requirements and technical preferences. 

### Agent 2: The Planner Agent

**We will now use Structured Outputs, and include a description of the fields**

In [10]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query")

In [11]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [12]:
INSTRUCTIONS = f"""
You are  a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(name='Planner Agent', instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=WebSearchPlan)
result = await Runner.run(planner_agent, task)
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='To identify leading AI agent frameworks that are trending or gaining traction in 2026.', query='most popular AI agent frameworks 2026'), WebSearchItem(reason='To discover recent developments, reviews, and rankings of AI frameworks in 2026.', query='AI frameworks comparison 2026'), WebSearchItem(reason='To find out the key features and advantages of popular AI agent frameworks in that year.', query='top AI agent frameworks features 2026'), WebSearchItem(reason='To explore emerging technologies and innovations in AI agent frameworks for 2026.', query='innovations in AI agent frameworks 2026'), WebSearchItem(reason='To gather expert opinions and analysis on the future of AI agent frameworks.', query='expert analysis AI frameworks 2026')])

### Agent 3: The Writer Agent

In [13]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed.
Aim for 5-10 pages of content, at least 1000 words.
"""

class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: str = Field(description="Suggested topics to research further")


writer_agent = Agent(name='Writer Agent', instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=ReportData)

### Agent 4: The email agent

In [14]:
@function_tool 
def send_email_tool(subject: str, text_body: str, html_body: str)-> str:
    """
    Send out an email with the given subject and body to all sales prospects.

    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """

    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    
    return "Email sent successfully"

In [15]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(name='Email Agent', instructions=INSTRUCTIONS, tools=[send_email_tool], model=MODEL_NAME)

### Orchestrate by Code

The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

In [19]:
async def run_searches(query: str):
    print("Planning searches....")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches

    print(f"Will perform: {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print(f"Finished Searching")
    return results


async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

The next 2 functions write a report and email it

In [20]:
async def write_report(query: str, search_results: list[str]):
    print(f"Thinking about report....")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_email(report: ReportData):
    print(f"Writting email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email Sent")
    return result.final_output

Showtime!

In [21]:
query = "Most popular AI Agent framework in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)
    print("Hooray!")

Starting research...
Planning searches....
Will perform: 5 searches
Finished Searching
Thinking about report....
Finished writing report
Writting email...
Push: Subject: Comprehensive Report on Popular AI Agent Frameworks in 2026

Dear Team,\n\nPlease find below our comprehensive report on the most popular AI agent frameworks in 2026. This detailed analysis includes an overview of leading frameworks, current trends, challenges, and future prospects in the AI landscape.\n\n---\n\n## Comprehensive Report on the Most Popular AI Agent Frameworks in 2026\n\n### Introduction\nThe landscape of artificial intelligence (AI) has witnessed rapid transformation in recent years, particularly concerning AI agent frameworks...\n\n### Overview of Top AI Agent Frameworks\n#### 1. LangGraph  \nLangGraph has established itself as a powerhouse in the AI agent framework space...\n\n#### 2. Microsoft Agent Framework  \nEntwined deeply within the Microsoft ecosystem, the Microsoft Agent Framework offers an a